# MVA · NB06 — Case Evidence Synthesis
**v1.0.0 · public-only · claim-boundary audit · no new patient-data access**

Purpose: consolidate frozen public outputs from NB01–NB05A3 into a submission-safe evidence ledger. This notebook does not reopen the patient VCF, does not invent F0 results, and does not assign PS3/BS3.

In [ ]:
import os
import json
from pathlib import Path
import pandas as pd
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    ROOT = Path(os.environ.get("MVA_ROOT", "/path/to/MVA_Hackathon_2026"))
except Exception:
    ROOT=Path('.')
PUBLIC=ROOT/'03_PUBLIC_ARTIFACTS'
VERSION='v1_0_0'
print('ROOT:',ROOT)

## 1. Require final public handoffs

In [ ]:
required=[
 'NB01B_HANDOFF.json',
 'NB01C_BUB1B_PHASE_HANDOFF.json',
 'NB02B_BUBR1_MUSCLE_HANDOFF_v1_0_3.json',
 'NB04D_NEPHROCALCINOSIS_HANDOFF.json',
 'NB05A2_MANUAL_CURATION_HANDOFF_v1_0_2.json',
 'NB05A3_F0_IMPLEMENTATION_HANDOFF_v1_0_2.json'
]
for name in required:
    p=PUBLIC/name
    if not p.exists(): raise FileNotFoundError(p)
print('Required final public handoffs: PASS')

## 2. Load evidence ledger

In [ ]:
ledger=pd.read_csv(PUBLIC/'NB06_CASE_EVIDENCE_LEDGER_v1_0_0.csv')
display(ledger)
print('\nStatus counts:')
display(ledger['status'].value_counts().rename_axis('status').reset_index(name='n'))

## 3. Hard claim-boundary assertions

In [ ]:
phase = json.loads((PUBLIC/'NB01C_BUB1B_PHASE_HANDOFF.json').read_text())
muscle = json.loads((PUBLIC/'NB02B_BUBR1_MUSCLE_HANDOFF_v1_0_3.json').read_text())
neph = json.loads((PUBLIC/'NB04D_NEPHROCALCINOSIS_HANDOFF.json').read_text())
cur = json.loads((PUBLIC/'NB05A2_MANUAL_CURATION_HANDOFF_v1_0_2.json').read_text())
f0 = json.loads((PUBLIC/'NB05A3_F0_IMPLEMENTATION_HANDOFF_v1_0_2.json').read_text())
assert phase['phase_status']=='PHASE_UNRESOLVED'
assert muscle['mitochondrial_recurrence']['Mito_translation']['call']=='CONTEXT_SPECIFIC'
assert muscle['mitochondrial_recurrence']['OXPHOS']['call']=='CONTEXT_SPECIFIC'
assert neph['result']=='NO_STRONG_SECONDARY_CANDIDATE_IN_FROZEN_CODING_SPLICE_PANEL'
assert cur['current_ps3_bs3_assigned'] is False
assert f0['F0V']['PS3_BS3_assigned_now'] is False
assert f0['shared']['NB05B_gate']=='blocked until real F0 data exist'
print('Claim-boundary assertions: PASS')

## 4. ACMG decision states

In [ ]:
acmg=pd.read_csv(PUBLIC/'NB06_ACMG_DECISION_STATES_v1_0_0.csv')
display(acmg)

## 5. Final submission-safe conclusion

**BUB1B L737*/N1002K remains the leading mechanistic research hypothesis, not a resolved molecular diagnosis. Parental phase and direct validated N1002K function are the decisive remaining evidence.**

## Stop rule

No NB05B result analysis is permitted until real F0 data exist. No drug selection. No automatic PS3/BS3 or LP assignment.